# MLflow Eval 📊

모델과 애플리케이션을 프로덕션에 배포하기 전에 평가할 수 있어야 합니다.  
이를 위해 MLflow의 평가 프레임워크를 사용할 수 있습니다 🙌  
이 프레임워크는 프롬프트와 기대 답변을 여러 scorer(평가자)에 통과시켜 모델이 기대한 대로 답변하는지 확인합니다.  
프롬프트와 기대 답변은 MLflow가 관리하는 데이터셋에 저장되므로, 평가 데이터의 버전을 관리하고 실험과 함께 시간에 따른 변화를 추적할 수 있습니다.

여기서는 두 가지 유형의 평가를 테스트합니다.
- **단순/기능 scorer**(`is_concise`): LLM 출력이 글자 수 제한을 지키는지 테스트합니다. 추가 LLM 호출이 필요 없어 빠르고 비용이 적게 듭니다
- **LLM as judge** scorer(`summary_quality`): LLM이 생성된 답변이 대화에 얼마나 잘 대응하는지 평가합니다

시스템 프롬프트가 실제로 얼마나 효과적인지도 함께 테스트할 수 있도록 백엔드 엔드포인트를 평가합니다 😉

## 설정
먼저 클라이언트를 설정하겠습니다. vLLM에서 호스팅되는 Llama 3.2 모델을 가리키는 OpenAI 호환 클라이언트를 사용하고, 평가 결과를 tracking 서버에 기록하도록 MLflow를 설정합니다.

In [ ]:
import mlflow
from mlflow.entities import AssessmentSource, AssessmentSourceType
from mlflow.genai.scorers import scorer
from mlflow.genai.judges import make_judge
from openai import OpenAI
from typing import Literal
import os, re, json
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [ ]:
LLM_ENDPOINT = "http://llama-32-predictor.ai501.svc.cluster.local:8080"
MODEL_NAME = "llama32"

client = OpenAI(
    base_url=LLM_ENDPOINT + "/v1",
    api_key="no-key-required", #일단은
)

이제 MLflow가 tracking 서버를 가리키도록 설정하겠습니다. 모든 평가 실행이 한곳에 묶이도록 experiment 이름도 지정합니다.

In [ ]:
MLFLOW_TRACKING_URI = "https://mlflow.redhat-ods-applications.svc.cluster.local:8443"

os.environ["MLFLOW_TRACKING_AUTH"] = "kubernetes"
os.environ["MLFLOW_TRACKING_INSECURE_TLS"] = "true"

NAMESPACE_PATH = "/run/secrets/kubernetes.io/serviceaccount/namespace"
if os.path.exists(NAMESPACE_PATH):
    with open(NAMESPACE_PATH) as f:
        os.environ["MLFLOW_WORKSPACE"] = f.read().strip()

SA_TOKEN_PATH = "/run/secrets/kubernetes.io/serviceaccount/token"
if os.path.exists(SA_TOKEN_PATH):
    with open(SA_TOKEN_PATH) as f:
        os.environ["MLFLOW_TRACKING_TOKEN"] = f.read().strip()

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("summarization")

print(f"MLflow tracking URI: {MLFLOW_TRACKING_URI}")
print(f"Workspace: {os.environ.get('MLFLOW_WORKSPACE', 'not set')}")

# 데이터셋 📖

평가 데이터는 MLflow가 관리하는 데이터셋에 저장합니다. 이렇게 하면 데이터의 버전을 관리하고, 여러 실행에서 재사용하고, 각 평가가 정확히 어떤 데이터로 실행되었는지 추적할 수 있습니다.  
기존 데이터셋을 가져와 새 샘플을 병합합니다. 각 샘플에는 `inputs`(대화 메시지)와 `expectations`(최대 글자 수처럼 점수를 매길 기준)가 있습니다.

## 데이터셋에 샘플 추가하기

In [ ]:
eval_dataset = mlflow.genai.get_dataset(
    name="eval",
)

print(json.dumps(eval_dataset.to_dict(), indent=2))

In [ ]:
new_samples = [                                                                                                                                                                                                                        
    {                                                                                                                                                                                                                                  
        "inputs": {                                                                                                                                                                                                                  
            "messages": [
                {"role": "user", "content": "Artificial intelligence is transforming healthcare by enabling faster diagnostics and personalized treatment plans."},
            ],
            "session_id": "session-001",
        },
        "expectations": {"length": 150},
    },
    {
        "inputs": {
            "messages": [
                {"role": "user", "content": "The Eiffel Tower was built in 1889 and stands 330 metres tall in Paris, France."},
                {"role": "assistant", "content": "The Eiffel Tower, built in 1889, is a 330-metre landmark in Paris. 🗼"},
                {"role": "user", "content": "What year was it built again?"},
            ],
            "session_id": "session-002",
        },
        "expectations": {"length": 100},
    },
]

eval_dataset.merge_records(new_samples)

# 업데이트된 레코드 수를 확인하기 위해 다시 가져오기
eval_dataset = mlflow.genai.get_dataset(name="eval")
print(f"Dataset now has {len(eval_dataset.to_dict()['records'])} record(s)")

이제 데이터셋에 평가 샘플이 들어 있으며, 각 샘플은 입력과 기대 답변을 짝지은 것입니다. 데이터 저장과 버전 관리는 MLflow가 처리해 줍니다 📦

# Scorer

이제 모델의 응답을 평가하는 데 사용할 scorer를 정의하겠습니다. 서로 보완적인 두 가지 방식을 사용합니다. 하나는 단순하고 빠른 방식이고, 다른 하나는 LLM을 심사위원(judge)으로 사용하는 방식입니다.

## 단순하게 시작하기

단순한 기능 검사로 시작하면 빠를 뿐 아니라, 실행에 LLM 호출이 필요 없으므로 비용도 줄일 수 있습니다.  
다음과 같은 항목을 테스트할 수 있습니다.
- **응답 길이**: 출력이 글자 수 제한 이내인가?
- **구조 검사**: 출력이 유효한 JSON인가, 또는 필수 필드를 포함하는가?
- **동작 신호**: 모델이 기대한 키워드를 반환하거나 정해진 형식을 따르는가?

우리의 사용 사례는 요약기이므로, 자연스러운 첫 번째 검사는 **응답이 기대 글자 수 제한보다 짧은가?**입니다.

In [ ]:
@scorer
def is_concise(outputs: str, expectations: dict) -> bool:
    """응답이 글자 수 제한 이내인가?"""
    if not outputs:
        return False
    return len(outputs) <= expectations.get("length", 300)

## LLM as Judge 🧑‍⚖️

다음으로 MLflow의 `make_judge`를 사용해 LLM-as-judge scorer를 만듭니다.  
좋은 응답이 어떤 모습인지 설명하는 지침을 제공하면, scorer는 생성된 각 답변을 `yes`/`no`로 판정합니다.  
여기서는 Llama 3.2 모델을 응답자와 심사위원으로 모두 사용하므로, 모델이 자기 자신의 응답을 평가하게 됩니다(💸💸💸).

In [ ]:
os.environ["OPENAI_API_KEY"] = "no-key-required"

summary_quality_judge = make_judge(
    name="summary_quality",
    instructions=(
        "Evaluate the quality of the GENERATED_RESPONSE given the CONVERSATION history.\n\n"
        "{{ inputs }}\n"
        "{{ outputs }}\n\n"
        "Is the response a concise, accurate, and helpful reply to the conversation?"
    ),
    feedback_value_type=Literal["yes", "no"],
    model="openai:/llama32",
    base_url="http://llama-32-predictor.ai501.svc.cluster.local:8080/v1/chat/completions",
    extra_headers={"Authorization": "Bearer no-key-required"},
)

## 평가 실행하기

백엔드에 있는 프롬프트와 코드 그대로를 평가하기 위해 백엔드 엔드포인트를 대상으로 평가를 실행합니다.

In [ ]:
def send_request(payload, url):
    import httpx
    import json
    full_response = ""

    with httpx.Client(timeout=None) as http_client:
        with http_client.stream("POST", url, json=payload) as response:
            if response.status_code != 200:
                error_body = response.read().decode()
                raise RuntimeError(f"Backend returned {response.status_code}: {error_body}")
            for line in response.iter_lines():
                if line.startswith("data: "):
                    try:
                        data = json.loads(line[len("data: "):])
                        full_response += data.get("delta", "")
                    except json.JSONDecodeError:
                        continue

    return full_response

def predict_backend_summarize(messages: list, session_id: str = None, **kwargs) -> str:
    from urllib.parse import urljoin
    backend_url = "http://canopy-backend:8000"
    endpoint_to_test = "/summarization/chat"

    url = urljoin(backend_url, endpoint_to_test)
    payload = {
        "messages": messages,
        "session_id": session_id,
    }
    return send_request(payload, url)

In [ ]:
print("Running evaluation with our latest prompt...")
eval_v1 = mlflow.genai.evaluate(
    data=eval_dataset,      # <- MLflow에서 호스팅되는 데이터셋 사용
    predict_fn=predict_backend_summarize,
    scorers=[is_concise, summary_quality_judge],
)

print("\n── Evaluation Results: Prompt v1 ──")
for metric, value in sorted(eval_v1.metrics.items()):
    bar = "█" * int((value if isinstance(value, float) else 0) * 20)
    print(f"  {metric:<35} {f'{value:.0%}' if isinstance(value, float) else value}  {bar}")

print(f"\n✓ Results saved. View per-example breakdowns in the Evaluations tab.")
print(f"  → {MLFLOW_TRACKING_URI}")

정리하면, MLflow의 평가 프레임워크로 단순 기능 scorer와 LLM as judge를 모두 사용해 백엔드 엔드포인트를 평가했습니다.  
결과는 MLflow에 기록되므로 시간에 따른 평가 실행을 비교하고, 프롬프트나 모델을 변경했을 때 성능이 정확히 어디에서 개선되거나 저하되는지 확인할 수 있습니다.  
이 구성을 바탕으로 백엔드와 모델을 프로덕션에 배포하기 전에 테스트하는 평가 워크플로를 만들 수 있습니다 👏